# **Vehicle speed calibration**

> This notebook export the calibration parameter ```METER_PER_PIXEL```in a .txt file stored as ```/results/traffic_analysis/meter_per_pixel.txt```. It provides a simple GUI utilitary tool to perform pixel-to-meter calibration based on known real-world distances (implementation assisted by ChatGPT). Once the video is selected and started, right-click to pause on the current frame, then draw the length of a visible vehicle located in the lower zone of the image.
The first click records the starting point, and the second click records the ending point of the selected vehicle length.

## 1. Libraries import

In [ ]:

import cv2 # Import OpenCV for video processing
from pathlib import Path # Import Path for file path management
import math
import numpy as np # Import NumPy for numerical operations

## 2. Path Configuration

In [ ]:
PROJECT_ROOT = Path.cwd().parent.resolve() # Root path of the project

VIDEO_PATH = PROJECT_ROOT / "data_processed" / "test" / "demo_video.mp4" # Path input video

## 3. Calibration script

In [ ]:
## PLEASE RIGHT-CLICK ON THE VIDEO TO PAUSE/RESUME ##
## AND LEFT-CLICK TO SELECT 2 POINTS FOR CALIBRATION ON EACH CAR ##

# ============================================================
# CALIBRATION PARAMETERS
# ============================================================

KNOWN_DISTANCE_M = 4.5   # average real-world car length (meters)
N_CARS = 5              # number of cars used for calibration
LOWER_ZONE_RATIO = 0.3  # zone basse de l'image pour la calibration

# Default parameters (uncomment to override)
# KNOWN_DISTANCE_M = 4.5    
# N_CARS = 5
# LOWER_ZONE_RATIO = 0.3

# total number of clicks required (2 points per car)
TOTAL_POINTS = 2 * N_CARS


# ============================================================
# GLOBAL STATE VARIABLES
# ============================================================

points = []   # list of clicked points [(x, y), ...]
mpp = []      # list of meters-per-pixel values (one per car)
paused = False  # video pause state


# ============================================================
# MOUSE CALLBACK FUNCTION
# ============================================================

def on_mouse(event, x, y, flags, param):
    """
    Mouse interactions:
    - RIGHT CLICK  : pause / resume video
    - LEFT CLICK   : select a point for calibration
    """

    global paused

    # --------------------------------------------------------
    # RIGHT CLICK → toggle pause / resume
    # --------------------------------------------------------
    if event == cv2.EVENT_RBUTTONDOWN:
        paused = not paused

    # --------------------------------------------------------
    # LEFT CLICK → select calibration point
    # --------------------------------------------------------
    if event == cv2.EVENT_LBUTTONDOWN:

        # stop accepting clicks once enough points are collected
        if len(points) >= TOTAL_POINTS:
            return

        # store the clicked point
        points.append((x, y))

        # ----------------------------------------------------
        # every pair of points corresponds to one car
        # ----------------------------------------------------
        if len(points) % 2 == 0:

            # retrieve the last two points
            (x1, y1), (x2, y2) = points[-2], points[-1]

            # pixel distance between the two points
            dpx = math.hypot(x2 - x1, y2 - y1)

            # avoid division by zero
            if dpx > 0:
                # compute meters-per-pixel for this car
                mpp.append(KNOWN_DISTANCE_M / dpx)

                # print intermediate result
                print(f"m/px = {mpp[-1]:.6f}")


# ============================================================
# VIDEO INITIALIZATION
# ============================================================

cap = cv2.VideoCapture(VIDEO_PATH)

# safety check
if not cap.isOpened():
    raise RuntimeError("cannot open the video")

# create display window
cv2.namedWindow("calib")

# attach mouse callback to the window
cv2.setMouseCallback("calib", on_mouse)

frame = None  # current video frame


# ============================================================
# MAIN LOOP
# ============================================================

while True:

    # --------------------------------------------------------
    # read a new frame only if video is not paused
    # --------------------------------------------------------
    if not paused or frame is None:
        ret, frame = cap.read()
        if not ret:
            break

    # --------------------------------------------------------
    # image geometry
    # --------------------------------------------------------
    H, W = frame.shape[:2]

    # lower zone threshold (used for consistency with speed computation)
    Y_MIN = int(H * (1 - LOWER_ZONE_RATIO))

    # copy frame for visualization (do NOT draw on original)
    vis = frame.copy()

    # --------------------------------------------------------
    # display number of selected points
    # --------------------------------------------------------
    cv2.putText(
        vis,
        f"{len(points)}/{TOTAL_POINTS}",  # current / total
        (20, 110),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    # --------------------------------------------------------
    # draw lower zone reference line
    # --------------------------------------------------------
    cv2.line(
        vis,
        (0, Y_MIN),
        (W, Y_MIN),
        (255, 0, 0),
        2
    )

    # --------------------------------------------------------
    # draw already measured segments (one per car)
    # --------------------------------------------------------
    for i in range(0, len(points), 2):
        if i + 1 < len(points):
            cv2.line(
                vis,
                points[i],
                points[i + 1],
                (0, 255, 0),
                2
            )

    # --------------------------------------------------------
    # pause indicator
    # --------------------------------------------------------
    if paused:
        cv2.putText(
            vis,
            "paused",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )

    # --------------------------------------------------------
    # display frame
    # --------------------------------------------------------
    cv2.imshow("calib", vis)

    # --------------------------------------------------------
    # exit conditions:
    # - ESC key
    # - enough cars measured
    # --------------------------------------------------------
    key = cv2.waitKey(20) & 0xFF
    if key == 27 or len(mpp) == N_CARS:
        break


# ============================================================
# CLEANUP
# ============================================================

cap.release()
cv2.destroyAllWindows()


# ============================================================
# FINAL CALIBRATION RESULT
# ============================================================

# compute final meters-per-pixel as the average over all cars
METER_PER_PIXEL = np.mean(mpp) if len(mpp) > 0 else None

if len(mpp) > 0:
    print("METERS_PER_PIXEL =", METER_PER_PIXEL)
else:
    print("No valid calibration measurement")


No valid calibration measurement


In [ ]:
# ============================================================
# SAVE CALIBRATION RESULT
# ============================================================

# Output directory
CALIB_DIR = PROJECT_ROOT / "results" / "traffic_analysis"
CALIB_DIR.mkdir(parents=True, exist_ok=True)

CALIB_FILE = CALIB_DIR / "meter_per_pixel.txt"

if METER_PER_PIXEL is not None:
    with open(CALIB_FILE, "w") as f:
        f.write(f"METER_PER_PIXEL = {METER_PER_PIXEL:.8f}\n")

    print(f"Calibration saved to: {CALIB_FILE}")
else:
    print("Calibration not saved: METER_PER_PIXEL is None")